# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*\

A page is flagged for review if it's ranking in a position where clicks are known to be weak (position worse than 10, where CTR has already dropped below 0.35 in my bucket check) and it's actively trending worse, not just sitting still. I'm deliberately excluding the noisy tail bucket (position > 100, n=1,991) from triggering the strongest reason code, since that bucket's numbers are driven by near-zero-impression pages, not real signal  those get downgraded to a weaker/monitor-only code instead.

The idea: don't flag every low-ranking page flag the ones that are both underperforming and getting worse, since those are the ones actively losing ground and worth someone's attention now, versus pages that are just chronically low but stable (lower priority) or too sparse in data to trust (exclude/monitor only).

In [24]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)


In [25]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)

df = ds.to_pandas()
print(df.shape)

(9841378, 30)


In [26]:
tail_bucket = df[
    (df['gsc_avg_position'] > 100) & (df['gsc_avg_position'] <= 500)
]
print(tail_bucket[['gsc_impressions', 'gsc_clicks']].describe())

       gsc_impressions   gsc_clicks
count      1991.000000  1991.000000
mean         16.440984     0.016575
std          66.403664     0.131579
min           1.000000     0.000000
25%           1.000000     0.000000
50%           2.000000     0.000000
75%           5.000000     0.000000
max        1190.000000     2.000000


In [27]:
import pandas as pd
df['pos_bucket'] = pd.cut(df['gsc_avg_position'], bins=[0,3,10,20,100,500])
df['CTR'] = (df['gsc_clicks'] / df['gsc_impressions']) * 100

bucket_table = df.groupby('pos_bucket')['CTR'].agg(['mean', 'count'])
print(bucket_table)

                mean    count
pos_bucket                   
(0, 3]      0.491821   564173
(3, 10]     0.347264  1456122
(10, 20]    0.276991   519223
(20, 100]   0.127814   906363
(100, 500]  0.630059     1991


/tmp/ipykernel_24392/1453039466.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table = df.groupby('pos_bucket')['CTR'].agg(['mean', 'count'])


In [28]:
df.sort_values(by=['report_date', 'content_hash_id'], ascending=True, inplace=True)

df['trend_direction'] = df.groupby('content_hash_id')['gsc_avg_position'].diff(1).fillna(0)

In [29]:
df.groupby('pos_bucket')['trend_direction'].agg(['mean', 'count'])

/tmp/ipykernel_24392/1832354741.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('pos_bucket')['trend_direction'].agg(['mean', 'count'])


,mean,count
pos_bucket,,
"(0, 3]",-2.425004,564173
"(3, 10]",-1.433693,1456122
"(10, 20]",-0.355909,519223
"(20, 100]",5.508778,906363
"(100, 500]",61.144268,1991




**CTR VS Position behind the CTR-fix logic**
- CONFIRMED: CTR declines cleanly and monotonically from 0.492 (position 0-3) to 0.128 (position 20-100) across 99.95% of rows confirming the CTR-fix logic pattern. The final bucket (position 100-500, n=1,991) shows mean CTR of 0.630, which looks like a reversal, but checking gsc_impressions within that bucket shows a median of just 2 impressions (75th percentile = 5) these are near-zero-traffic pages where a single click produces an extreme CTR ratio (e.g. 1 click / 2 impressions = 50%). Mean clicks in this bucket is 0.017, meaning almost all rows actually have 0 clicks the high mean CTR is an artifact of dividing by tiny denominators on a handful of rows, not a genuine behavioral reversal.

**Trend_direction vs position bucket (staleness-adjacent)**
- CONFIRMED Bucketed by gsc_avg_position, computed mean trend_direction per bucket (negative = improving position, positive = worsening). Clean pattern across 99.95% of rows: top-ranked pages (0-10) trend improving (-2.43 to -1.43), mid-ranked pages (10-20) are roughly stable (-0.36), and poorly-ranked pages (20-100) are actively declining further (+5.51). The (100,500] bucket (n=1,991) shows an extreme +61.14 same small, near-zero-traffic bucket flagged in Signal 1, likely volatile position readings on barely-measured pages rather than a real trend.
Verdict: CONFIRMED

In [30]:
!git clone https://github.com/ocedev112/flyrank_oce_dev
%cd flyrank_oce_dev/

Cloning into 'flyrank_oce_dev'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 143 (delta 54), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.92 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/flyrank_oce_dev/flyrank_oce_dev/flyrank_oce_dev


In [31]:
import os
print(os.getcwd())

/content/flyrank_oce_dev/flyrank_oce_dev/flyrank_oce_dev


In [32]:
import os
os.makedirs('work/outputs', exist_ok=True)
print(os.path.exists('work/outputs'))

True


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [33]:
import os
import numpy as np
import pandas as pd

os.makedirs('work/outputs', exist_ok=True)

#Encoded Rule
position = df['gsc_avg_position'].values
trend = df['trend_direction'].values
impressions = df['gsc_impressions'].values

nan_mask = pd.isna(position)
low_impressions_mask = (~nan_mask) & (impressions < 10)
low_conf_mask = (~nan_mask) & (position > 100)
healthy_mask = (~nan_mask) & (position <= 10) & (~low_impressions_mask)
declining_mask = (~nan_mask) & (position > 10) & (position <= 100) & (trend > 0) & (~low_impressions_mask)

position_weakness = np.clip((position - 10) / 90, 0, 1)
trend_worsening = np.clip(trend / 100, 0, 1)
score = np.round(position_weakness * 60 + trend_worsening * 40, 4)
score[nan_mask | low_conf_mask | healthy_mask | low_impressions_mask] = 0.0

df['action_score'] = score

#Reason Code
reason_code = np.full(len(df), 'WEAK_BUT_STABLE', dtype=object)
action = np.full(len(df), 'monitor', dtype=object)

reason_code[nan_mask] = 'NO_DATA'
action[nan_mask] = 'monitor'

reason_code[low_impressions_mask] = 'LOW_CONFIDENCE_SIGNAL'
action[low_impressions_mask] = 'monitor'

reason_code[low_conf_mask] = 'LOW_CONFIDENCE_SIGNAL'
action[low_conf_mask] = 'monitor'

reason_code[healthy_mask] = 'HEALTHY'
action[healthy_mask] = 'no_action'

reason_code[declining_mask] = 'DECLINING_UNDERPERFORMER'
action[declining_mask] = 'review_priority'

df['reason_code'] = reason_code
df['action'] = action

ranked = df.sort_values('action_score', ascending=False)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(ranked['reason_code'].value_counts())
print(ranked.head(10)[['content_hash_id','gsc_avg_position','trend_direction','gsc_impressions','action_score','reason_code','action']])

del position, trend, reason_code, action, position_weakness, trend_worsening, score, nan_mask, low_conf_mask, healthy_mask, declining_mask, ranked
import gc
gc.collect()

reason_code
NO_DATA                     6230317
LOW_CONFIDENCE_SIGNAL       1463851
HEALTHY                     1343651
DECLINING_UNDERPERFORMER     460041
WEAK_BUT_STABLE              343518
Name: count, dtype: int64
                  content_hash_id  gsc_avg_position  trend_direction  \
7651946  content_485f4462f81878ba         99.153846        99.153846   
7648470  content_a3728f3736d7b3c4         98.882353        98.215686   
8586028  content_222ad8b72e0409ca         98.750000        91.350000   
8583588  content_4c8f60c8875f8cc6         99.345313        89.507889   
8584956  content_450304ab9cd8fcbe         95.785714        95.341270   
7648023  content_45c7886717eb649d         98.454545        89.787879   
8586668  content_dc9a374d34ff3ab7         98.000000        90.428571   
8584874  content_52b47afacb073069         96.000000        93.750000   
7648313  content_1b46fa0fc079e148         96.863636        92.017483   
8583981  content_efc52466147baebf         95.090909        94.

0

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

- Row 0  review_priority: DECLINING_UNDERPERFORMER: position 99.2, trend +99.2 (worsening sharply), 13 impressions, 0 clicks. Ranks almost at the edge of page 10, actively declining, and getting zero clicks despite being crawled. What would make this wrong: 13 impressions is still fairly thin  a couple of unlucky search sessions could account for the "decline," and the position swinging from presumably page-1-ish to 99 within a month is a big jump that deserves a manual look at content_created_date (may be a young page still settling).

- Row 1  review_priority: DECLINING_UNDERPERFORMER: position 98.9, trend +98.2, 17 impressions, 0 clicks. Same profile as row 0  near-total ranking collapse with no click activity. What would make this wrong: same thin-impression caveat; also worth checking if this page shares a topic/keyword cluster with a newer page that may be cannibalizing its rank (a keyword_hash_id check would confirm).

- Row 2  review_priority: DECLINING_UNDERPERFORMER: position 98.8, trend +91.4, 12 impressions, 0 clicks. Consistent pattern  high-magnitude decline, no clicks. What would make this wrong: 12 impressions over a month is low enough that a single day's crawl anomaly could be driving most of this trend value.

- Row 3  review_priority: DECLINING_UNDERPERFORMER: position 99.3, trend +89.5, 640 impressions, 4 clicks (CTR 0.625%). This is your strongest, most credible pick  real traffic volume, not noise, and it's genuinely declining while getting meaningfully more impressions than typical page-100 content. What would make this wrong: unlikely to be a data-noise issue given the volume; more likely a real ranking drop  worth checking content_updated_date/last_optimized_date to see if a recent edit coincided with the decline (correlation, not proof, but a reasonable next check).

- Row 4  review_priority: DECLINING_UNDERPERFORMER: position 95.8, trend +95.3, 28 impressions, 0 clicks. Slightly better position than others but still deep decline. What would make this wrong: 28 impressions is on the low end; check whether this page belongs to a content_type (e.g. category/hub page) where position 95 is structurally normal rather than a true regression.

- Row 5  review_priority: DECLINING_UNDERPERFORMER: position 98.5, trend +89.8, 11 impressions, 0 clicks. What would make this wrong: very thin sample (11 impressions)  same low-confidence caveat as rows 0-2.

- Row 6  review_priority: DECLINING_UNDERPERFORMER: position 98.0, trend +90.4, 10 impressions, 0 clicks. Sits right at your impression cutoff (10)  borderline case. What would make this wrong: this is literally at the threshold you set for "trustworthy"  worth flagging as the shakiest inclusion in the top 10 precisely because it barely cleared the bar.

- Row 7  review_priority: DECLINING_UNDERPERFORMER: position 96.0, trend +93.8, 13 impressions, 0 clicks. What would make this wrong: same thin-sample caveat as rows 0, 2, 5.

- Row 8  review_priority: DECLINING_UNDERPERFORMER: position 96.9, trend +92.0, 22 impressions, 0 clicks. What would make this wrong: moderate sample size, somewhat more trustworthy than the 10-13 impression rows, but zero clicks across 22 impressions could still be normal variance for a low-CTR topic area, not necessarily "decline."

- Row 9  review_priority: DECLINING_UNDERPERFORMER: position 95.1, trend +94.7, 11 impressions, 0 clicks. What would make this wrong: same thin-sample caveat.

In [34]:
top10 = pd.read_csv('work/outputs/baseline_action_score.csv', nrows=10)
print(top10[['content_hash_id', 'gsc_avg_position', 'trend_direction', 'gsc_impressions', 'gsc_clicks', 'CTR', 'action_score', 'reason_code', 'action']].to_string())

            content_hash_id  gsc_avg_position  trend_direction  gsc_impressions  gsc_clicks    CTR  action_score               reason_code           action
0  content_485f4462f81878ba         99.153846        99.153846               13           0  0.000       99.0974  DECLINING_UNDERPERFORMER  review_priority
1  content_a3728f3736d7b3c4         98.882353        98.215686               17           0  0.000       98.5412  DECLINING_UNDERPERFORMER  review_priority
2  content_222ad8b72e0409ca         98.750000        91.350000               12           0  0.000       95.7067  DECLINING_UNDERPERFORMER  review_priority
3  content_4c8f60c8875f8cc6         99.345313        89.507889              640           4  0.625       95.3667  DECLINING_UNDERPERFORMER  review_priority
4  content_450304ab9cd8fcbe         95.785714        95.341270               28           0  0.000       95.3270  DECLINING_UNDERPERFORMER  review_priority
5  content_45c7886717eb649d         98.454545        89.787879  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

9 of my top 10 picks are weak. Impressions range from 10 to 640, but most sit between 10 and 28 with 0 clicks. My cutoff was impressions less than 10, and row 6 sits at exactly 10, so it barely cleared the bar rather than being a genuinely strong signal.

Row 3 is the one pick I trust. 640 impressions and 4 clicks is a real sample size, and a decline at that volume is more likely to be genuine than noise.

Next time I would raise the cutoff to somewhere around 30 to 50 impressions. 10 was too low given how sparse this data is.

Leakage check

No future windows leaked in. All features come from report_date within March 2026. trend_direction is a diff computed within each content_hash_id, ordered by report_date, so it only uses information up to that day. I did not touch the sample table, which covers June 2026.

No outcome flags leaked in. The rule only uses gsc_avg_position, gsc_clicks, gsc_impressions, and word_count. I left out last_optimized_date and optimization_eligible_date on purpose, since those reflect whether action was already taken, not something I would know before flagging a page.

One thing worth flagging: content_updated_date is not part of the rule, but I mentioned it a few times in the top 10 notes as something worth checking manually. If its timestamp does not line up cleanly with report_date, it could carry the same risk as the word_count issue from Week 3.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.